In [1]:
!pip install -q fastparquet
!pip install -q statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 46.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, ADIDA, IMAPA, WindowAverage, CrostonSBA, CrostonOptimized

%matplotlib inline

#####################################################
# Импортируем .py файлы с уже написанными функциями #
#####################################################

import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

# функции для расчёта метрик
from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
    summarize_metrics_by_segments,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [3]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

In [4]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'lumpy' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(912713, 3)

### Фильтруем ряды

In [5]:
from split_utils import filter_long_series, split_train_val_test

real_demand_filtered = filter_long_series(
    real_demand,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    min_train_points=35,
    id_col="unique_id",
)

eval_df = real_demand_filtered[["unique_id", "ds", "y"]].sort_values(["unique_id", "ds"])

### Обучение

In [7]:
# 1. Создаем список с моделями
models = [   
    Naive(alias='Naive'),  
    SeasonalNaive(season_length=7, alias='SeasonalNaive7'),    
    WindowAverage(window_size=7, alias='WindowAvg_7'), 
    OptimizedTheta(season_length=7, alias='OptTheta'),             
    AutoETS(alias='AutoETS'),                                                   
    TSB(alpha_d=0.15, alpha_p=0.15, alias='TSB_015_015'),   
    CrostonOptimized(alias='CrostonOpt'),
    CrostonSBA(alias='CrostonSBA'),
    ADIDA(alias='ADIDA'),
    IMAPA(alias='IMAPA'),
]

# обучаем только на train, val - подбор наилучших параметров, test - итоговое качество метрик
sf = StatsForecast(models=models, freq="D", n_jobs=1)
cv_frozen = sf.cross_validation(
    df=eval_df,
    h=14,
    step_size=14,
    n_windows=4,   # 1 val + 3 test
    refit=False,   # не переобучаем параметры
)

### Собираем прогнозы

In [37]:
val_cutoff = pd.Timestamp("2025-08-05")
test_cutoffs = [
    pd.Timestamp("2025-08-19"),
    pd.Timestamp("2025-09-02"),
    pd.Timestamp("2025-09-16"),
]

val_pred = cv_frozen[cv_frozen["cutoff"] == val_cutoff].copy()
test_pred = cv_frozen[cv_frozen["cutoff"].isin(test_cutoffs)].copy()

cutoff_to_window = {c: i + 1 for i, c in enumerate(test_cutoffs)}
test_pred["test_window"] = test_pred["cutoff"].map(cutoff_to_window)

### Считаем метрики

In [39]:
from metrics import DEFAULT_METRICS, get_model_columns, compute_metrics_per_window, summarize_metrics

val_model_cols = get_model_columns(
    val_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
val_metrics_per_window = compute_metrics_per_window(val_pred, val_model_cols, DEFAULT_METRICS)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)

test_model_cols = get_model_columns(
    test_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
test_metrics_per_window = compute_metrics_per_window(test_pred, test_model_cols, DEFAULT_METRICS)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)

print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
ADIDA,1.195106,2.249316,154.069850,111.818203
AutoETS,1.213965,2.455460,155.247758,113.582696
CrostonOpt,1.260633,2.278350,152.105629,117.949081
CrostonSBA,1.272920,2.283347,152.316449,119.098715
IMAPA,1.186470,2.245553,154.119407,111.010119
Naive,1.387514,2.941539,147.946383,129.820482
OptTheta,1.246176,2.494537,156.133291,116.596486
SeasonalNaive7,1.361666,2.957397,151.372892,127.402102
TSB_015_015,1.201209,2.247937,153.343507,112.389210



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
ADIDA,1.210814,2.629790,154.013493,109.593991
AutoETS,1.265911,2.806365,155.630384,114.581422
CrostonOpt,1.297146,2.742330,152.072156,117.389693
CrostonSBA,1.280876,2.702826,152.238819,115.920794
IMAPA,1.210738,2.628658,154.193029,109.584590
Naive,1.377928,3.164397,146.599135,124.712000
OptTheta,1.269629,2.773968,156.124510,114.909266
SeasonalNaive7,1.410860,3.305931,148.641481,127.690369
TSB_015_015,1.212251,2.612279,153.249403,109.719051


In [40]:
test_pred.to_csv("lumpy_forecast.csv")